In [ ]:
import numpy as np

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings


embeddings= HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader,UnstructuredWordDocumentLoader

In [ ]:
try:
    docx_loader = Docx2txtLoader("C:\\Users\\nimaa\\Desktop\\Projects\\RAGUdemy\\0-DataIngestion\\data\\word_files\\cmd_report.docx") 
    docs = docx_loader.load()
    
    print(f"Loaded {len(docs)} documents!")
except Exception as e:
    print(f"Error loading doc: {e}")

# Extract the text content from the Document objects
texts = [doc.page_content for doc in docs]

# Pass the list of strings to the embedding model
embedded_vectors = embeddings.embed_documents(texts)

print(f'Embedding count: {len(embedded_vectors)}')

# Safety check before printing indices
if len(embedded_vectors) > 0:
    print(f'First embedding (first 5 dims): {embedded_vectors[0][:100]}...\n')
    if len(embedded_vectors) > 1:
        print(f'Second embedding: {embedded_vectors[1][:5]}...')
    else:
        print("Note: Only one document/vector generated.")


In [ ]:
print(docs[0].page_content)

In [ ]:
sentences = [
    "the cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in python",
    "python is my favorite programming language"

]

In [ ]:
import numpy as np

def cosine_similarity(vec1,vec2):
    
    
    dot_product = np.dot(vec1,vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return dot_product / (norm_a * norm_b)

In [ ]:
sentences_embeddings = embeddings.embed_documents(sentences)
sentences_embeddings

In [ ]:
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        similarity  =   cosine_similarity(sentences_embeddings[i],sentences_embeddings[j])
        
        print(f'similarity : {similarity:.3f}\n')

In [ ]:
document = [
    "langchain is a framework for deeloping appllications powered by language models",
    "python is a high-level programming language",
    "machin learning is a subset of artificial intelligence",
    "embeddings convert text into numberical vetors",
    "The weather today is sunny and warm"
]

query = "What is langchain?"

In [ ]:
#semantic search

def semantic_search (query,documents,embedding_models , top_k = 3):
    
    query_embedding = embedding_models.embed_documents(query)
    doc_embeddings = embedding_models.embed_documents(documents)
    
    
    similarities = []
    
    
    for i, doc_emb in enumerate(doc_embeddings):
        similarity = cosine_similarity(query_embedding, doc_emb)

        similarities.append((float(similarity), documents[i]))

    similarities.sort(key=lambda x: x[0], reverse=True)

    return similarities[:top_k]

In [ ]:



def cosine_similarity(vec1, vec2):
    vec1 = np.array(vec1, dtype=np.float64).flatten()
    vec2 = np.array(vec2, dtype=np.float64).flatten()
    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)
    return float(dot_product / (norm_a * norm_b))


# ---------------------------------------------------------------------------
# Semantic search
# ---------------------------------------------------------------------------
documents = [
    "langchain is a framework for developing applications powered by language models",
    "python is a high-level programming language",
    "a subset of artificial intelligence which has been popularized in the recent years is machine learning ",
    "embeddings convert text into numerical vectors",
    "The weather today is sunny and warm",
]

query = "is the weather bad today?"


def semantic_search(query, documents, embedding_model, top_k=3):
    query_embedding = embedding_model.embed_query(query)
    doc_embeddings = embedding_model.embed_documents(documents)

    similarities = []

    for i, doc_emb in enumerate(doc_embeddings):
        similarity = cosine_similarity(query_embedding, doc_emb)
        similarities.append((similarity, documents[i]))

    similarities.sort(key=lambda x: x[0], reverse=True)
    return similarities[:top_k]


result = semantic_search(query, documents, embeddings)
print(result)


[(0.6781414151171289, 'The weather today is sunny and warm'), (0.05029983025799937, 'python is a high-level programming language'), (0.021368123958773386, 'embeddings convert text into numerical vectors')]
